# Reproduce: fetal HPO terms and gene ranking in prenatal exome sequencing
Run All downloads the public inputs, reruns every analysis step, redraws the figures and compares every number with the committed `outputs/`.

In [ ]:
import os, json, shutil, glob
os.makedirs('work/reference', exist_ok=True)
for f in glob.glob('outputs/*.json'):
    shutil.copy(f, 'work/reference/')


In [ ]:
import sys, json, subprocess
import pandas as pd
subprocess.run(['bash', 'scripts/download.sh'], check=True)
for s in ['fu_cases.py', 'rank.py', 'gaps.py', 'explore.py']:
    subprocess.run([sys.executable, s], cwd='scripts', check=True, capture_output=True)

## Primary and external cohorts (PLAN outcomes)

In [ ]:
R = json.load(open('outputs/rank_summary.json'))
pd.DataFrame({c: {k: v for k, v in R[c].items() if k.startswith(('top10', 'median', 'fetuses', 'mcnemar', 'wilcoxon'))} for c in ['fu2022', 'external', 'graham2026', 'smogavec2022', 'aogs2026']}).T

## Annotation gaps and leakage check

In [ ]:
G = json.load(open('outputs/gaps.json')); {k: v for k, v in G.items() if k != 'by_branch_exact_2026'}

## Double extraction: agreement and sensitivity analysis (PLAN amendments 2026-10-15)

In [ ]:
for s in ['agreement.py', 'sensitivity.py']:
    subprocess.run([sys.executable, s], check=True, cwd='scripts', capture_output=True)
display(json.load(open('outputs/agreement.json')))
json.load(open('outputs/sensitivity.json'))

## Exploratory analyses X1 to X4 (post hoc, PLAN amendment)

In [ ]:
X = json.load(open('outputs/explore.json')); {k: v for k, v in X.items()}

In [ ]:
# Study flow schematic (figures/study_flow/)
from IPython.display import Image
subprocess.run([sys.executable, 'scripts/flowchart.py'], check=True)
Image('figures/study_flow/study_flow.png', width=900)

## Descriptive analyses, robustness analyses R1 to R7, figures

In [ ]:
for s in ['paper_analyses.py', 'review_analyses.py', 'review_r6.py', 'paper_figures.py']:
    r = subprocess.run([sys.executable, s], cwd='scripts', check=True, capture_output=True, text=True)
print(r.stdout)

In [ ]:
# compare every regenerated number with the committed outputs
bad = []
for f in sorted(glob.glob('work/reference/*.json')):
    if json.load(open(f)) != json.load(open(os.path.join('outputs', os.path.basename(f)))):
        bad.append(os.path.basename(f))
print('REPRODUCED' if not bad else f'DIFFERENT: {bad}')